## CNN 기반 이미지 분류하기

Colab 실행 환경 설정

In [ ]:
# from google.colab import userdata
# key = userdata.get('hf-api')

# from huggingface_hub import login
# login(token=key)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
print(f"현재 작업 폴더: {os.getcwd()}")
print("폴더 내 파일 목록:", os.listdir())
!ls

import os
target_folder = '/content/drive/MyDrive/08_Data'  # 학습 데이터 있는 폴더 설정
os.chdir(target_folder)
print(f"현재 작업 폴더: {os.getcwd()}")
print("폴더 내 파일 목록:", os.listdir())

# from google.colab import files
# data_to_load = files.upload()
# !unzip -q dogs-vs-cats.zip -d ./dogs-vs-cats

이미지 데이터 정규화

In [1]:
import torchvision.transforms as transforms

# 이미지 변환 규칙을 순서대로 정의
transform = transforms.Compose([
	transforms.Resize((64, 64)),  # 크기 맞추기
	transforms.ToTensor(),        # 텐서 데이터타입으로 변환
	transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5)) # 밝기/대비 표준화
])


데이터셋 정의

In [2]:
import os
from tqdm import tqdm
from PIL import Image
from torch.utils.data import Dataset

# 파이토치 Dataset 클래스를 상속받아 우리만의 데이터 학습 모델 정의
class CatsVsDogsDataset(Dataset):
	def __init__(self, root_dir, transform=None):
		self.root_dir = root_dir
		self.transform = transform
		self.image_files = os.listdir(root_dir) # 데이터 폴더 안의 모든 파일 이름을 리스트로 저장
		
	def __len__(self):
		return len(self.image_files)

	def __getitem__(self, idx): # 특정 인덱스(idx)의 데이터(이미지와 레이블)를 요청받았을 때 실행
		img_name = self.image_files[idx] 				 # idx에 해당하는 파일 이름을 획득
		img_path = os.path.join(self.root_dir, img_name) # 파일 이름과 폴더 경로를 합쳐 전체 파일 경로를 생성
		image = Image.open(img_path).convert('RGB') 	 # 'PIL' 라이브러리로 이미지를 열고, RGB 형식으로 통일
		
		label = 0 if 'cat' in img_name else 1   # 파일 이름에 'cat'이 있으면 레이블을 0, 아니면(dog이면) 1로 설정

		if self.transform: # 변환 규칙(transform)이 있다면, 이미지를 규칙에 맞게 가공
			image = self.transform(image)
			
		return image, label  # 가공된 이미지와 정답(레이블)을 함께 반환


데이터 로더 정의

In [3]:
from torch.utils.data import DataLoader, random_split

module_path = os.getcwd() # Jupyter 환경에서는 __file__이 없으므로 현재 작업 디렉토리 사용
train_data_path = os.path.join(module_path, 'dogs-vs-cats', 'train')
full_dataset = CatsVsDogsDataset(root_dir=train_data_path, transform=transform)

train_size = int(0.8 * len(full_dataset)) # 전체 데이터를 훈련용(80%)과 검증용(20%)으로 분할
val_size = len(full_dataset) - train_size
train_dataset, val_dataset = random_split(full_dataset, [train_size, val_size])

batch_size = 1024  # 데이터 배치 크기
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True) # 학습 데이터를 무작위로 섞어(shuffle=True) 모델이 패턴을 외우지 못하게 함
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)  # 검증 데이터는 순서대로 데이터를 제공

print(f"훈련 데이터: {len(train_dataset)}개")
print(f"검증 데이터: {len(val_dataset)}개")


훈련 데이터: 20000개
검증 데이터: 5000개


CNN 기반 이미지 분류 모델 아키텍처 정의

In [4]:
import torch.nn as nn
import torch.nn.functional as F

class SimpleResCNN(nn.Module):
	def __init__(self):
		super(SimpleResCNN, self).__init__()
		self.conv1 = nn.Conv2d(3, 16, 3, padding=1)
		self.pool = nn.MaxPool2d(2, 2)
		
		self.conv2 = nn.Conv2d(16, 16, 3, padding=1)  # 첫 번째 conv (채널 유지)
		self.conv3 = nn.Conv2d(16, 16, 3, padding=1)  # 두 번째 conv (채널 유지)
		
		self.fc1 = nn.Linear(16 * 16 * 16, 128)
		self.fc2 = nn.Linear(128, 2) # Classification layers

	def forward(self, x):
		x = F.relu(self.conv1(x))  	# (3,64,64) → (16,64,64)
		x = self.pool(x)           	# (16,64,64) → (16,32,32)
		skip_connect = x  			# skip connection을 위해 입력 저장
		out = self.conv2(x)
		out = F.relu(out)    		# conv + relu
		out = self.conv3(out)       
		out = out + skip_connect  	# Skip connection: F(x) + x
		out = F.relu(out)         	# 최종 activation
		x = self.pool(out)  # (16,32,32) → (16,16,16)
		x = x.view(-1, 16 * 16 * 16)
		x = self.fc1(x)
		x = F.relu(x) # 분류를 위한 pooling 및 FC layer
		x = self.fc2(x)
		return x

model = SimpleResCNN()
print(model)


SimpleResCNN(
  (conv1): Conv2d(3, 16, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (pool): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  (conv2): Conv2d(16, 16, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (conv3): Conv2d(16, 16, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (fc1): Linear(in_features=4096, out_features=128, bias=True)
  (fc2): Linear(in_features=128, out_features=2, bias=True)
)


모델 학습

In [ ]:
import torch.optim as optim, numpy as np, torch

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"사용 중인 디바이스: {device}")
if torch.cuda.is_available():	
	print(f"GPU 이름: {torch.cuda.get_device_name(0)}")
	print(f"GPU 메모리: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")

model = model.to(device) # 모델을 GPU로 이동

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)
num_epochs = 20 			# 에폭 횟수 설정
best_val_loss = np.inf 		# 최고 점수를 기록할 변수를 아주 큰 값으로 초기화

print("훈련과 검증을 시작합니다...")
for epoch in tqdm(range(num_epochs), desc="Epoch"): # 정해진 횟수만큼 전체 데이터셋을 반복 학습

	model.train() # 모델을 '학습 모드'로 전환
	running_loss = 0.0
	train_pbar = tqdm(train_loader, desc=f"Epoch {epoch+1} Training")
	for inputs, labels in train_pbar:
		inputs, labels = inputs.to(device), labels.to(device)  # 데이터를 GPU로 이동
		optimizer.zero_grad()      
		outputs = model(inputs)    
		loss = criterion(outputs, labels) 
		loss.backward()            
		optimizer.step()           
		running_loss += loss.item()
		
		train_pbar.set_postfix({'loss': f'{loss.item():.4f}', 'avg_loss': f'{running_loss/(train_pbar.n+1):.4f}'})

	print(f'[{epoch + 1}] 훈련 손실: {running_loss / len(train_loader):.3f}')

	model.eval() # 모델을 '평가 모드'로 전환 (학습 기능 일시 정지)
	val_loss = 0.0
	val_pbar = tqdm(val_loader, desc=f"Epoch {epoch+1} Validation")
	with torch.no_grad(): # 정답을 봐도 학습(가중치 업데이트)하지 않도록 설정
		for inputs, labels in val_pbar:
			inputs, labels = inputs.to(device), labels.to(device)  # 데이터를 GPU로 이동
			outputs = model(inputs)
			loss = criterion(outputs, labels)
			val_loss += loss.item()
			
			val_pbar.set_postfix({'loss': f'{loss.item():.4f}', 'avg_loss': f'{val_loss/(val_pbar.n+1):.4f}'})
	
	current_val_loss = val_loss / len(val_loader)
	print(f'[{epoch + 1}] 검증 손실: {current_val_loss:.3f}')

	if current_val_loss < best_val_loss: # 최고 점수 모델 저장
		best_val_loss = current_val_loss
		torch.save(model.state_dict(), 'best_cnn_model.pth') # 현재 모델의 상태 저장
		print(f'모델을 "best_cnn_model.pth"에 저장했습니다')

print('훈련 종료!')


사용 중인 디바이스: cuda
GPU 이름: NVIDIA GeForce RTX 3080 Laptop GPU
GPU 메모리: 8.0 GB
훈련과 검증을 시작합니다...


Epoch 1 Training: 100%|██████████| 20/20 [00:49<00:00,  2.48s/it, loss=0.6947, avg_loss=0.7095]


[1] 훈련 손실: 0.709


Epoch:   5%|▌         | 1/20 [00:59<18:58, 59.94s/it]

[1] 검증 손실: 0.687
모델을 "best_cnn_model.pth"에 저장했습니다


...

[2] 검증 손실: 0.666
모델을 "best_cnn_model.pth"에 저장했습니다


Epoch 3 Training: 100%|██████████| 20/20 [00:34<00:00,  1.74s/it, loss=0.6452, avg_loss=0.6411]


[3] 훈련 손실: 0.641


Epoch:  15%|█▌        | 3/20 [02:25<13:13, 46.67s/it]

[3] 검증 손실: 0.601
모델을 "best_cnn_model.pth"에 저장했습니다


모델 추론(Inference Model)

In [ ]:
best_model = SimpleResCNN().to(device)  # 모델을 GPU로 이동
best_model.load_state_dict(torch.load('best_cnn_model.pth'))
best_model.eval() # 평가 모드로 설정

image_path = os.path.join(module_path, 'new_image.jpg')  # 절대 경로로 변경
image = Image.open(image_path).convert('RGB')
input_tensor = transform(image) # 훈련 때와 똑같은 변환 규칙(transform)을 적용
input_batch = input_tensor.unsqueeze(0).to(device)  # 모델은 이미지 묶음(배치) 단위로 입력을 받으므로, 한 장의 이미지를 묶음으로 변환

classes = ('cat', 'dog') # 정답 종류를 정의(0번째는 'cat', 1번째는 'dog')
with torch.no_grad(): 
	output = best_model(input_batch)

_, predicted_idx = torch.max(output, 1) # 모델의 출력값 중 가장 높은 점수를 받은 인덱스를 획득
predicted_class = classes[predicted_idx.item()] # 해당 인덱스에 맞는 클래스 이름을 가져옴

print(f'\n최종 예측 결과: "{image_path}" 이미지는 "{predicted_class}" 입니다.')

import matplotlib.pyplot as plt
plt.imshow(image)
plt.title(f'Predicted: {predicted_class}')
plt.axis('off')
plt.show()


최종 예측 결과: "f:\projects\Book_AI_foundation\5_DL_foundation\new_image.jpg" 이미지는 "cat" 입니다.
